In [ ]:
!pip install pymongo
import pymongo

client = pymongo.MongoClient(
    "mongodb://localhost:27017/"
)

db = client["movie_analysis"]

collection = db["movies"]

collection.insert_one({"test": "success"})

print("Connected and inserted!")

In [ ]:
import pandas as pd
import pymongo

client = pymongo.MongoClient(
    "mongodb://localhost:27017/"
)

db = client["movie_analysis"]
collection = db["movies"]


collection.delete_many({})


df = pd.read_csv("clean_movies.csv")

def transform(row):
    return {
        "tconst": row["tconst"],
        "title": row["primaryTitle"],
        "year": int(row["startYear"]),
        "genres": row["genres"].split(","),
        "imdb": {
            "rating": float(row["averageRating"]),
            "votes": int(row["numVotes"])
        }
    }

documents = [transform(row) for _, row in df.iterrows()]

collection.insert_many(documents)

print("DONE")

In [ ]:
import pandas as pd
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")

db = client["movie_analysis"]
collection = db["rotten"]

df = pd.read_csv("clean_rotten_tomatoes.csv")

def transform(row):
    return {
        "title": row["title"],
        "year": int(row["year"]) if pd.notna(row["year"]) else None,
        "ratings": {
            "audience": float(row["audience_score"]) if pd.notna(row["audience_score"]) else None,
            "critics": float(row["critics_score"]) if pd.notna(row["critics_score"]) else None
        }
    }

documents = [transform(row) for _, row in df.iterrows()]

collection.insert_many(documents)

print(f"Inserted {len(documents)} documents into Rotten collection")

In [ ]:
import pandas as pd
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")

db = client["movie_analysis"]
collection = db["oscars"]

df = pd.read_csv("clean_oscar.csv")

df.columns = df.columns.str.strip().str.lower()

def transform(row):
    return {
        "year": int(row["year"]) if pd.notna(row["year"]) else None,
        "film": row["film"],
        "filmid": row["filmid"]
    }

documents = [transform(row) for _, row in df.iterrows()]

collection.insert_many(documents)

print(f"Inserted {len(documents)} documents into Oscars collection")

In [ ]:
pipeline = [
    {
        "$lookup": {
            "from": "rotten",
            "let": { "title": "$title", "year": "$year" },
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$and": [
                                { "$eq": ["$title", "$$title"] },
                                { "$eq": ["$year", "$$year"] }
                            ]
                        }
                    }
                }
            ],
            "as": "rotten"
        }
    },
    {
        "$lookup": {
            "from": "oscars",
            "localField": "tconst",
            "foreignField": "filmid",
            "as": "oscars"
        }
    },
    {
        "$addFields": {
            "rotten": { "$arrayElemAt": ["$rotten", 0] },
            "oscar": { "$gt": [ { "$size": "$oscars" }, 0 ] }
        }
    },
    {
        "$project": {
            "oscars": 0
        }
    },
    {
        "$out": "merged_movies"
    }
]

In [ ]:
db.movies.aggregate(pipeline)

In [ ]:
db.merged_movies.find_one()